# AI Model Creation on Raspberry Pi 5 🍓🤖

This notebook shows you how to create, fine-tune, and deploy AI models on your Pi 5 using LocalAI.

## Available Methods:
1. **Fine-tuning existing models** (Recommended for Pi 5)
2. **Creating custom Ollama models**
3. **Converting HuggingFace models**
4. **Training small models from scratch** (Limited on Pi 5)

In [ ]:
# Install required packages
!pip install transformers datasets accelerate peft huggingface_hub ollama
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

## Method 1: Fine-tune a Small Model (Recommended)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import Dataset
import torch

# Load a small, Pi 5-friendly base model
model_name = "microsoft/DialoGPT-small"  # 117M parameters
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(model_name)

# Add pad token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Loaded {model_name} with {model.num_parameters():,} parameters")

In [ ]:
# Create your training data
training_data = [
    "Hello, how can I help you today?",
    "I'm a helpful AI assistant running on Raspberry Pi 5!",
    "What would you like to know about LocalAI?",
    "I can help you with various tasks and questions."
]

# Tokenize the data
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

dataset = Dataset.from_dict({"text": training_data})
tokenized_dataset = dataset.map(tokenize_function, batched=True)

print(f"📊 Training dataset size: {len(tokenized_dataset)}")

In [ ]:
# Configure training for Pi 5 (lightweight settings)
training_args = TrainingArguments(
    output_dir="/home/jovyan/training/models/my-pi5-model",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=1,  # Small batch for Pi 5
    gradient_accumulation_steps=4,   # Simulate larger batch
    learning_rate=5e-5,
    save_steps=50,
    save_total_limit=2,
    logging_steps=10,
    dataloader_num_workers=0,  # Single-threaded for Pi
    fp16=False,  # CPU training
    report_to=None,  # Disable wandb/tensorboard
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

print("🚀 Starting training... This may take a while on Pi 5")

## Method 2: Create Ollama Model

In [ ]:
# Create a custom Ollama model
import ollama

# Create a Modelfile
modelfile = '''
FROM llama2:7b
SYSTEM You are a helpful AI assistant running on Raspberry Pi 5. You are knowledgeable about LocalAI and edge computing.
PARAMETER temperature 0.7
PARAMETER top_p 0.9
'''

# Save the Modelfile
with open('/home/jovyan/training/Modelfile', 'w') as f:
    f.write(modelfile)

print("📝 Created custom Modelfile for Ollama")
print("Run: ollama create my-pi5-assistant -f /home/jovyan/training/Modelfile")

## Method 3: Convert HuggingFace Model to GGUF

In [ ]:
# Install llama.cpp for conversion
!git clone https://github.com/ggerganov/llama.cpp.git /tmp/llama.cpp
!cd /tmp/llama.cpp && make

# Convert your fine-tuned model to GGUF format
conversion_script = """
#!/bin/bash
cd /tmp/llama.cpp
python convert-hf-to-gguf.py /home/jovyan/training/models/my-pi5-model \
  --outfile /home/jovyan/models/my-pi5-model.gguf \
  --outtype q4_0
"""

with open('/home/jovyan/training/convert_model.sh', 'w') as f:
    f.write(conversion_script)

!chmod +x /home/jovyan/training/convert_model.sh
print("🔄 Created conversion script: /home/jovyan/training/convert_model.sh")

## Method 4: Deploy to LocalAI

In [ ]:
# Create LocalAI model configuration
model_config = """
name: my-pi5-model
backend: llama
parameters:
  model: my-pi5-model.gguf
  threads: 4
  temperature: 0.7
  top_p: 0.9
  context_size: 2048
  f16: true
"""

with open('/home/jovyan/models/my-pi5-model.yaml', 'w') as f:
    f.write(model_config)

print("⚙️ Created LocalAI model configuration")
print("Your model will be available at: http://localhost:8080/v1/models")

## Test Your Model

In [ ]:
import requests
import json

# Test your model via LocalAI API
def test_model(prompt, model_name="my-pi5-model"):
    url = "http://localai:8080/v1/chat/completions"
    
    payload = {
        "model": model_name,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 100,
        "temperature": 0.7
    }
    
    try:
        response = requests.post(url, json=payload, timeout=30)
        if response.status_code == 200:
            return response.json()['choices'][0]['message']['content']
        else:
            return f"Error: {response.status_code} - {response.text}"
    except Exception as e:
        return f"Connection error: {str(e)}"

# Test your model
test_prompt = "Hello! Tell me about yourself."
response = test_model(test_prompt)
print(f"🤖 Model Response: {response}")